# LeetCode 1118 — Number of Days in a Month bằng Python

**Link đề:** https://leetcode.com/problems/number-of-days-in-a-month/
**Độ khó:** Easy · **Chủ đề:** Math

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def numberOfDays(self, year: int, month: int) -> int:
        ...
```

## Mục tiêu học
1. Nắm chính xác **quy tắc năm nhuận** của lịch Gregory (chia hết cho 4, trừ thế kỷ không chia hết cho 400).
2. Luyện điều kiện logic `and`, `or`, `not` và toán tử `%` qua một quy tắc có ngoại lệ lồng nhau.
3. Biết 4 cách cài đặt: chuỗi `if/elif` → **bảng tra cứu** (list/tuple) → thư viện chuẩn `calendar`/`datetime` → công thức thuần toán học.
4. Trace từng nhánh, chứng minh tính đúng, phân tích Time/Space, kiểm thử đối chiếu với thư viện chuẩn.
5. Tự giải các biến thể: ngày thứ mấy trong năm (#1154), số ngày giữa hai ngày (#1360).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — tháng này có bao nhiêu ngày?

Cho năm `year` và tháng `month`, trả về số ngày của tháng đó.

| year | month | Output | Lý do |
|---:|---:|---:|---|
| 1992 | 7 | 31 | tháng 7 luôn có 31 ngày |
| 2000 | 2 | 29 | 2000 chia hết cho 400 → năm nhuận |
| 1900 | 2 | 28 | 1900 chia hết cho 100 nhưng không chia hết cho 400 → không nhuận |

**Constraints (theo đề gốc):** `1583 <= year <= 2100`, `1 <= month <= 12`.

(Năm 1583 là năm trọn vẹn đầu tiên sau khi lịch Gregory được áp dụng vào cuối năm 1582 — nên quy tắc nhuận Gregory áp dụng cho mọi năm trong đề.)

## 2. Kiến thức nền: số ngày mỗi tháng

| Tháng | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Số ngày | 31 | 28/29 | 31 | 30 | 31 | 30 | 31 | 31 | 30 | 31 | 30 | 31 |

- Tháng **30 ngày**: 4, 6, 9, 11.
- Tháng **2**: 28 ngày, năm nhuận 29 ngày.
- Còn lại (1, 3, 5, 7, 8, 10, 12): **31 ngày**.

Mẹo "nắm tay": đếm trên các đốt và khe ngón tay — đốt lồi là 31 ngày, khe lõm là 30 (hoặc tháng 2). Tháng 7 và 8 liền nhau đều 31.

In [1]:
days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
print("Tổng số ngày năm thường:", sum(days))
print("Tháng 30 ngày:", [m for m in range(1, 13) if days[m - 1] == 30])
print("Tháng 31 ngày:", [m for m in range(1, 13) if days[m - 1] == 31])

Tổng số ngày năm thường: 365
Tháng 30 ngày: [4, 6, 9, 11]
Tháng 31 ngày: [1, 3, 5, 7, 8, 10, 12]


Cell trên dùng **list comprehension** `[biểu_thức for x in dãy if điều_kiện]`: tạo list mới từ các phần tử thỏa điều kiện — viết gọn của vòng `for` + `append`.

## 3. Quy tắc năm nhuận (lịch Gregory)

Một năm là **nhuận** (leap year) khi:
1. Chia hết cho 4, **và**
2. **Không** chia hết cho 100, **trừ khi** chia hết cho 400.

Viết thành điều kiện: `(year % 4 == 0 and year % 100 != 0) or year % 400 == 0`.

| Năm | `%4` | `%100` | `%400` | Nhuận? |
|---:|:---:|:---:|:---:|:---:|
| 1992 | 0 | 92 | 392 | Có |
| 1900 | 0 | 0 | 300 | **Không** |
| 2000 | 0 | 0 | 0 | **Có** |
| 2023 | 3 | 23 | 23 | Không |
| 2100 | 0 | 0 | 100 | Không |

Vì sao có ngoại lệ? Một năm thiên văn dài ~365.2422 ngày. Cứ 4 năm thêm 1 ngày → trung bình 365.25 (dư). Bỏ nhuận ở năm thế kỷ → 365.24 (thiếu chút). Giữ lại nhuận ở năm chia hết 400 → 365.2425, rất sát.

In [2]:
for y in [1992, 1900, 2000, 2023, 2100, 2024]:
    print(f"{y}: %4={y % 4:>1}, %100={y % 100:>2}, %400={y % 400:>3}")
print("Độ dài năm trung bình theo Gregory:", 365 + 1/4 - 1/100 + 1/400)

1992: %4=0, %100=92, %400=392
1900: %4=0, %100= 0, %400=300
2000: %4=0, %100= 0, %400=  0
2023: %4=3, %100=23, %400= 23
2100: %4=0, %100= 0, %400=100
2024: %4=0, %100=24, %400= 24
Độ dài năm trung bình theo Gregory: 365.2425


### 3.1. Toán tử logic `and`, `or`, `not` và thứ tự ưu tiên

- `A and B`: đúng khi **cả hai** đúng.
- `A or B`: đúng khi **ít nhất một** đúng.
- `not A`: đảo giá trị.
- Ưu tiên: `not` > `and` > `or`. Nên luôn thêm **ngoặc** để rõ ý.

`x % k == 0` nghĩa là "`x` chia hết cho `k`".

In [3]:
def is_leap(year: int) -> bool:
    return (year % 4 == 0 and year % 100 != 0) or year % 400 == 0

for y in [1992, 1900, 2000, 2023, 2100, 2024]:
    print(y, "->", is_leap(y))

1992 -> True
1900 -> False
2000 -> True
2023 -> False
2100 -> False
2024 -> True


Một cách viết khác, theo đúng thứ tự "ngoại lệ lồng nhau" — dễ đọc khi mới học:

In [4]:
def is_leap_nested(year: int) -> bool:
    if year % 400 == 0:
        return True        # ngoại lệ của ngoại lệ
    if year % 100 == 0:
        return False       # năm thế kỷ thường
    return year % 4 == 0   # quy tắc cơ bản

assert all(is_leap(y) == is_leap_nested(y) for y in range(1583, 2101))
print("Hai cách viết is_leap khớp nhau cho mọi năm 1583..2100")

Hai cách viết is_leap khớp nhau cho mọi năm 1583..2100


`all(...)` trả `True` nếu **mọi** phần tử đều đúng. Ở trên ta đưa vào một **generator expression** (giống list comprehension nhưng dùng ngoặc tròn, không tạo list).

## 4. Cách 1 — Chuỗi `if / elif` (trực giác)

Dịch thẳng bảng ở mục 2 sang code:

In [5]:
def number_of_days_if(year: int, month: int) -> int:
    if month == 2:
        return 29 if is_leap(year) else 28
    elif month in (4, 6, 9, 11):
        return 30
    else:
        return 31

print(number_of_days_if(1992, 7))   # 31
print(number_of_days_if(2000, 2))   # 29
print(number_of_days_if(1900, 2))   # 28

31
29
28


`month in (4, 6, 9, 11)` kiểm tra `month` có nằm trong **tuple** `(4, 6, 9, 11)` không. Tuple giống list nhưng **không đổi được** (immutable), viết bằng ngoặc tròn. Với vài phần tử, `in` trên tuple là đủ nhanh; với tập lớn nên dùng `set` (`{4, 6, 9, 11}`) để tra cứu trung bình `O(1)`.

In [6]:
t = (4, 6, 9, 11)
print(6 in t, 7 in t)
s = {4, 6, 9, 11}
print(6 in s, 7 in s, type(t).__name__, type(s).__name__)

True False
True False tuple set


### Nút thắt của cách `if/elif`

Không phải tốc độ (đều `O(1)`), mà là **dễ sai khi gõ**: nhầm tháng 30/31, quên tháng 2 năm nhuận. Dữ liệu (số ngày) bị trộn vào logic (các nhánh). Cách sạch hơn: tách **dữ liệu** ra một bảng, logic chỉ còn "tra bảng + xử lý ngoại lệ tháng 2".

## 5. Cách 2 — Bảng tra cứu (lời giải chính)

Lưu số ngày năm thường trong một tuple 12 phần tử. Tháng `month` (1..12) ứng với **chỉ số** `month - 1` (vì chỉ số bắt đầu từ 0). Nếu là tháng 2 năm nhuận, cộng thêm 1.

In [7]:
DAYS = (31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31)
month = 2
print("month =", month, "-> chỉ số", month - 1, "-> DAYS[...] =", DAYS[month - 1])
print("True + 0 =", True + 0, "| False + 0 =", False + 0)

month = 2 -> chỉ số 1 -> DAYS[...] = 28
True + 0 = 1 | False + 0 = 0


Mẹo nhỏ: trong Python `bool` là lớp con của `int` — `True` bằng `1`, `False` bằng `0`. Nên `28 + (month == 2 and is_leap(year))` cộng đúng 1 khi cả hai điều kiện đúng. Tuy vậy, viết rõ bằng `if` vẫn dễ đọc hơn cho người mới.

In [8]:
def number_of_days_table(year: int, month: int) -> int:
    days = DAYS[month - 1]
    if month == 2 and is_leap(year):
        days += 1
    return days

for y, m in [(1992, 7), (2000, 2), (1900, 2), (2024, 2), (2023, 4)]:
    print((y, m), "->", number_of_days_table(y, m))

(1992, 7) -> 31
(2000, 2) -> 29
(1900, 2) -> 28
(2024, 2) -> 29
(2023, 4) -> 30


## 6. TRACE từng bước

| (year, month) | `DAYS[month-1]` | `month == 2`? | `is_leap(year)` | Cộng thêm | Kết quả |
|---|---:|:---:|:---:|:---:|---:|
| (1992, 7) | 31 | Không | (không xét) | 0 | 31 |
| (2000, 2) | 28 | Có | `2000 % 400 == 0` → True | 1 | 29 |
| (1900, 2) | 28 | Có | `1900 % 100 == 0`, `% 400 = 300` → False | 0 | 28 |
| (2024, 2) | 28 | Có | `2024 % 4 == 0`, `% 100 = 24` → True | 1 | 29 |

Lưu ý **đánh giá ngắn mạch** (short-circuit): trong `A and B`, nếu `A` sai thì Python **không tính** `B`. Nên với tháng 7, `is_leap` không hề được gọi.

In [9]:
def trace_number_of_days(year: int, month: int) -> int:
    base = DAYS[month - 1]
    print(f"({year}, {month}): DAYS[{month - 1}] = {base}")
    if month != 2:
        print("   không phải tháng 2 -> giữ nguyên", base)
        return base
    a, b4, c = year % 4 == 0, year % 100 != 0, year % 400 == 0
    leap = (a and b4) or c
    print(f"   tháng 2: chia hết 4? {a}, không chia hết 100? {b4}, chia hết 400? {c} -> nhuận? {leap}")
    result = base + (1 if leap else 0)
    print("   kết quả:", result)
    return result

for y, m in [(1992, 7), (2000, 2), (1900, 2), (2024, 2)]:
    _ = trace_number_of_days(y, m)

(1992, 7): DAYS[6] = 31
   không phải tháng 2 -> giữ nguyên 31
(2000, 2): DAYS[1] = 28
   tháng 2: chia hết 4? True, không chia hết 100? False, chia hết 400? True -> nhuận? True
   kết quả: 29
(1900, 2): DAYS[1] = 28
   tháng 2: chia hết 4? True, không chia hết 100? False, chia hết 400? False -> nhuận? False
   kết quả: 28
(2024, 2): DAYS[1] = 28
   tháng 2: chia hết 4? True, không chia hết 100? True, chia hết 400? False -> nhuận? True
   kết quả: 29


In [10]:
# Minh họa short-circuit: hàm có print để thấy khi nào được gọi
def noisy_leap(y):
    print("   ...đang gọi is_leap(", y, ")")
    return is_leap(y)

for m in [7, 2]:
    print("month =", m, "->", m == 2 and noisy_leap(2000))

month = 7 -> False
   ...đang gọi is_leap( 2000 )
month = 2 -> True


## 7. Code nộp LeetCode

In [11]:
class Solution:
    def numberOfDays(self, year: int, month: int) -> int:
        days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
        if month == 2 and ((year % 4 == 0 and year % 100 != 0) or year % 400 == 0):
            return 29
        return days[month - 1]

sol = Solution()
print(sol.numberOfDays(1992, 7), sol.numberOfDays(2000, 2), sol.numberOfDays(1900, 2))

31 29 28


## 8. Cách 3 — Thư viện chuẩn: `calendar` và `datetime`

### 8.1. `calendar.monthrange(year, month)`

Trả về tuple `(thứ của ngày 1, số ngày trong tháng)`; thứ được đánh số `0 = Thứ Hai ... 6 = Chủ Nhật`. Ta chỉ cần phần tử `[1]`. Module còn có `calendar.isleap(year)`. (Từ Python 3.12, phần tử đầu được in dạng hằng `calendar.TUESDAY` — bản chất vẫn là số `1`.)

In [12]:
import calendar

print(calendar.monthrange(2000, 2))     # (thứ của 1/2/2000, 29)
print(calendar.monthrange(1900, 2)[1])  # 28
print(calendar.isleap(2000), calendar.isleap(1900))

def number_of_days_calendar(year: int, month: int) -> int:
    return calendar.monthrange(year, month)[1]

print(number_of_days_calendar(1992, 7))

(calendar.TUESDAY, 29)
28
True False
31


### 8.2. `datetime.date` — ngày đầu tháng sau trừ ngày đầu tháng này

Hiệu của hai đối tượng `date` là một `timedelta`, thuộc tính `.days` cho số ngày. Tháng 12 phải nhảy sang tháng 1 **năm sau**.

In [13]:
from datetime import date

def number_of_days_datetime(year: int, month: int) -> int:
    start = date(year, month, 1)
    if month == 12:
        nxt = date(year + 1, 1, 1)
    else:
        nxt = date(year, month + 1, 1)
    return (nxt - start).days

print(date(2000, 3, 1) - date(2000, 2, 1))
print(number_of_days_datetime(2000, 2), number_of_days_datetime(2023, 12))

29 days, 0:00:00
29 31


Hai cách này ngắn và đáng tin cậy trong code thật. Khi phỏng vấn, người ta thường muốn thấy bạn **tự viết** quy tắc năm nhuận — nên vẫn cần nắm cách 2.

## 9. Cách 4 — Công thức thuần toán học (không bảng, không `if` theo tháng)

Có một công thức nổi tiếng cho số ngày của tháng `m` trong **năm thường**:

```text
days(m) = 28 + (m + m // 8) % 2 + 2 % m + 2 * (1 // m)
```

Giải thích từng mảnh:
- `(m + m // 8) % 2`: với `m <= 7`, `m // 8 = 0` → tháng lẻ được `1` (31 ngày), tháng chẵn `0`. Từ tháng 8 trở đi `m // 8 = 1` → **đảo** chẵn lẻ: tháng chẵn 8, 10, 12 được `1`. Đây là "cú nhảy" ở tháng 7–8.
- `2 % m`: bằng `0` với `m = 1, 2`, bằng `2` với `m >= 3` → nâng các tháng từ 3 trở đi lên mức 30/31.
- `2 * (1 // m)`: chỉ bằng `2` khi `m = 1` (bù lại cho tháng 1 vì `2 % 1 = 0`).

Rồi cộng 1 nếu `m == 2` và năm nhuận.

In [14]:
for m in range(1, 13):
    a, b5, c = (m + m // 8) % 2, 2 % m, 2 * (1 // m)
    print(f"m={m:>2}: (m+m//8)%2={a}, 2%m={b5}, 2*(1//m)={c} -> {28 + a + b5 + c}")

m= 1: (m+m//8)%2=1, 2%m=0, 2*(1//m)=2 -> 31
m= 2: (m+m//8)%2=0, 2%m=0, 2*(1//m)=0 -> 28
m= 3: (m+m//8)%2=1, 2%m=2, 2*(1//m)=0 -> 31
m= 4: (m+m//8)%2=0, 2%m=2, 2*(1//m)=0 -> 30
m= 5: (m+m//8)%2=1, 2%m=2, 2*(1//m)=0 -> 31
m= 6: (m+m//8)%2=0, 2%m=2, 2*(1//m)=0 -> 30
m= 7: (m+m//8)%2=1, 2%m=2, 2*(1//m)=0 -> 31
m= 8: (m+m//8)%2=1, 2%m=2, 2*(1//m)=0 -> 31
m= 9: (m+m//8)%2=0, 2%m=2, 2*(1//m)=0 -> 30
m=10: (m+m//8)%2=1, 2%m=2, 2*(1//m)=0 -> 31
m=11: (m+m//8)%2=0, 2%m=2, 2*(1//m)=0 -> 30
m=12: (m+m//8)%2=1, 2%m=2, 2*(1//m)=0 -> 31


In [15]:
def number_of_days_formula(year: int, month: int) -> int:
    m = month
    days = 28 + (m + m // 8) % 2 + 2 % m + 2 * (1 // m)
    return days + (1 if m == 2 and is_leap(year) else 0)

print([number_of_days_formula(2023, m) for m in range(1, 13)])
print([number_of_days_formula(2024, m) for m in range(1, 13)])

[31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
[31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]


Công thức này là một **câu đố thú vị** để luyện `%` và `//`, nhưng khó đọc — trong code thật nên dùng bảng tra cứu.

## 10. Vì sao đúng? — So sánh các cách và độ phức tạp

**Tính đúng của cách bảng tra cứu:** với `month != 2`, số ngày không phụ thuộc năm và bảng ghi đúng giá trị. Với `month == 2`, số ngày là `28 + [năm nhuận]`; `is_leap` cài đặt đúng 3 tầng quy tắc Gregory (4 → 100 → 400), mà đề đảm bảo `year >= 1583` nên lịch Gregory áp dụng. Ta còn kiểm chứng toàn bộ `518 × 12 = 6216` cặp `(year, month)` với `calendar` ở mục test.

| Cách | Ý tưởng | Time | Space | Nhận xét |
|---|---|---|---|---|
| 1. `if/elif` | rẽ nhánh theo tháng | `O(1)` | `O(1)` | Dễ hiểu, dễ gõ nhầm |
| 2. Bảng tra cứu | `DAYS[month-1]` + ngoại lệ tháng 2 | `O(1)` | `O(1)` (bảng 12 phần tử cố định) | **Nên dùng** |
| 3. `calendar` / `datetime` | thư viện chuẩn | `O(1)` | `O(1)` | Ngắn, đáng tin trong thực tế |
| 4. Công thức | `%`, `//` | `O(1)` | `O(1)` | Đố mẹo, khó đọc |

Mọi cách đều `O(1)` vì input chỉ là hai số và số tháng cố định 12.

## 11. Bộ test — mẫu, biên, đối chiếu toàn bộ miền input

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS {actual}")

cases = [
    ("mẫu 1992/7", 1992, 7, 31),
    ("mẫu 2000/2 (chia hết 400)", 2000, 2, 29),
    ("mẫu 1900/2 (thế kỷ không nhuận)", 1900, 2, 28),
    ("2024/2 (nhuận thường)", 2024, 2, 29),
    ("2023/2 (không nhuận)", 2023, 2, 28),
    ("2100/2 (biên trên, không nhuận)", 2100, 2, 28),
    ("1583/1 (biên dưới)", 1583, 1, 31),
    ("tháng 30 ngày", 2023, 11, 30),
    ("tháng 8 (sau tháng 7 vẫn 31)", 2023, 8, 31),
    ("tháng 12", 2023, 12, 31),
]
for name, y, m, exp in cases:
    check(name, Solution().numberOfDays(y, m), exp)

mẫu 1992/7                         PASS 31
mẫu 2000/2 (chia hết 400)          PASS 29
mẫu 1900/2 (thế kỷ không nhuận)    PASS 28
2024/2 (nhuận thường)              PASS 29
2023/2 (không nhuận)               PASS 28
2100/2 (biên trên, không nhuận)    PASS 28
1583/1 (biên dưới)                 PASS 31
tháng 30 ngày                      PASS 30
tháng 8 (sau tháng 7 vẫn 31)       PASS 31
tháng 12                           PASS 31


In [17]:
funcs = [number_of_days_if, number_of_days_table, number_of_days_calendar,
         number_of_days_datetime, number_of_days_formula, Solution().numberOfDays]
count = 0
for y in range(1583, 2101):
    for m in range(1, 13):
        expected = calendar.monthrange(y, m)[1]
        for f in funcs:
            assert f(y, m) == expected, (f.__name__, y, m)
        count += 1
print(f"Đã đối chiếu {count} cặp (year, month) — 6 cách đều khớp calendar")
print("Số năm nhuận trong 1583..2100:", sum(is_leap(y) for y in range(1583, 2101)))

Đã đối chiếu 6216 cặp (year, month) — 6 cách đều khớp calendar
Số năm nhuận trong 1583..2100: 126


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra chia hết cho 4

In [18]:
def wrong_leap_only_4(year):
    return year % 4 == 0

for y in [1900, 2100, 2000]:
    print(y, "sai:", wrong_leap_only_4(y), "| đúng:", is_leap(y))

1900 sai: True | đúng: False
2100 sai: True | đúng: False
2000 sai: True | đúng: True


**Vì sao sai:** bỏ qua ngoại lệ năm thế kỷ. 1900 và 2100 chia hết cho 4 nhưng **không** nhuận. Đề cố tình đưa ví dụ 1900.

### Lỗi 2: Đặt sai ngoặc / thứ tự ưu tiên

In [19]:
def alt_parentheses(year):
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)   # thực ra ĐÚNG!

def wrong_really(year):
    return year % 4 == 0 and year % 100 != 0 and year % 400 == 0     # SAI: and thay vì or

for y in [1992, 1900, 2000]:
    print(y, "| ngoặc khác nhưng đúng:", alt_parentheses(y), "| sai:", wrong_really(y), "| đúng:", is_leap(y))

1992 | ngoặc khác nhưng đúng: True | sai: False | đúng: True
1900 | ngoặc khác nhưng đúng: False | sai: False | đúng: False
2000 | ngoặc khác nhưng đúng: True | sai: False | đúng: True


**Nhận xét:** `4 and (không 100 or 400)` **tương đương** công thức chuẩn (vì chia hết 400 thì chắc chắn chia hết 4) — một cách viết đúng khác. Còn nối tất cả bằng `and` thì không năm nào thỏa (không thể vừa không chia hết 100 vừa chia hết 400) → luôn `False`.

### Lỗi 3: Lệch chỉ số bảng (quên `- 1`)

In [20]:
def wrong_index(year, month):
    days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    try:
        return days[month]            # SAI: tháng 1 lấy chỉ số 1 (= tháng 2)
    except IndexError as e:
        return f"IndexError ({e})"

print("tháng 1:", wrong_index(2023, 1), "| tháng 12:", wrong_index(2023, 12))

tháng 1: 28 | tháng 12: IndexError (list index out of range)


**Sửa:** `days[month - 1]`, hoặc thêm phần tử đệm ở đầu: `days = [0, 31, 28, ...]` để dùng `days[month]` trực tiếp.

### Lỗi 4: Cộng ngày nhuận cho mọi tháng của năm nhuận

In [21]:
def wrong_all_months(year, month):
    days = DAYS[month - 1]
    if is_leap(year):                 # SAI: thiếu điều kiện month == 2
        days += 1
    return days

print("2024/7 sai:", wrong_all_months(2024, 7), "| đúng:", number_of_days_table(2024, 7))

2024/7 sai: 32 | đúng: 31


## 13. Mẹo nhận dạng pattern

- Dữ liệu **cố định, ít phần tử** (12 tháng, 7 ngày trong tuần, bảng chữ La Mã) → **bảng tra cứu** (list/tuple/dict) thay cho chuỗi `if`.
- Quy tắc có **ngoại lệ lồng nhau** (4 → 100 → 400) → viết `if` theo thứ tự **ngoại lệ hẹp nhất trước**, hoặc công thức `(A and not B) or C`; luôn test từng tầng.
- Bài về ngày tháng → nhớ kiểm chứng với `calendar`/`datetime` khi được phép.
- Bài cùng họ: #1154 Day of the Year, #1185 Day of the Week, #1360 Number of Days Between Two Dates, #1507 Reformat Date.

## 14. Biến thể

### 14.1. Day of the Year (#1154): ngày thứ mấy trong năm?

Cộng số ngày các tháng **trước** tháng hiện tại, rồi cộng `day`.

In [22]:
def day_of_year(date_str: str) -> int:
    y, m, d = map(int, date_str.split("-"))      # "2019-02-10" -> 2019, 2, 10
    return sum(number_of_days_table(y, k) for k in range(1, m)) + d

print(day_of_year("2019-01-09"), day_of_year("2019-02-10"), day_of_year("2004-03-01"))

9 41 61


`map(int, list)` áp dụng `int` cho từng phần tử; gán `y, m, d = ...` là **unpacking** (tách 3 giá trị vào 3 biến).

### 14.2. Số ngày từ mốc `1971-01-01` (nền tảng cho #1360)

Đếm số ngày trọn các năm trước, các tháng trước, rồi ngày. Hiệu hai mốc = số ngày giữa hai ngày.

In [23]:
def days_since_1971(date_str: str) -> int:
    y, m, d = map(int, date_str.split("-"))
    total = sum(366 if is_leap(yy) else 365 for yy in range(1971, y))
    total += sum(number_of_days_table(y, k) for k in range(1, m))
    return total + d

def days_between(a: str, b_: str) -> int:
    return abs(days_since_1971(a) - days_since_1971(b_))

print(days_between("2019-06-29", "2019-06-30"), days_between("2020-01-15", "2019-12-31"))
print("đối chiếu datetime:", (date(2020, 1, 15) - date(2019, 12, 31)).days)

1 15
đối chiếu datetime: 15


## 15. Bài tập (tăng dần)

**Bài 1 — Tính tay:** các năm 1600, 1700, 1996, 1999 năm nào nhuận?

**Bài 2 — Dự đoán output:** `number_of_days_formula(2023, 8)` — tính từng mảnh `(m + m//8) % 2`, `2 % m`, `2 * (1 // m)`.

**Bài 3 — TODO:** viết `is_leap` theo kiểu `if` lồng nhau (ngoại lệ hẹp nhất trước).

In [24]:
def exercise_is_leap(year: int) -> bool:
    # TODO 1: chia hết 400 -> True
    # TODO 2: chia hết 100 -> False
    # TODO 3: còn lại: chia hết 4?
    pass

# assert exercise_is_leap(2000) and not exercise_is_leap(1900) and exercise_is_leap(2024)

**Bài 4 — Viết lại lời giải** bằng bảng có phần tử đệm `days = [0, 31, 28, ...]` để dùng `days[month]` trực tiếp.

In [25]:
def exercise_number_of_days(year: int, month: int) -> int:
    # TODO
    pass

# assert exercise_number_of_days(1992, 7) == 31
# assert exercise_number_of_days(2000, 2) == 29
# assert exercise_number_of_days(1900, 2) == 28

**Bài 5 — Biến thể:** đếm số tháng có đúng 31 ngày trong khoảng từ `(y1, m1)` đến `(y2, m2)` (bao gồm cả hai đầu).

In [26]:
def exercise_count_31_months(y1: int, m1: int, y2: int, m2: int) -> int:
    # TODO: duyệt từng (year, month) theo thứ tự
    pass

# assert exercise_count_31_months(2023, 1, 2023, 12) == 7

**Bài 6 — Mini challenge:** trả về **thứ trong tuần** của ngày `d/m/y` (`"Monday"`, …), **không** dùng `calendar`/`datetime`. Biết `1971-01-01` là thứ Sáu (Friday). Dùng `days_since_1971`.

In [27]:
def exercise_day_of_week(day: int, month: int, year: int) -> str:
    # TODO
    pass

# assert exercise_day_of_week(31, 8, 2019) == "Saturday"
# assert exercise_day_of_week(15, 8, 1993) == "Sunday"

### 15.1. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Áp 3 tầng quy tắc | 1600 % 400 = 0; 1700 % 400 = 100 | nhuận: 1600, 1996 |
| 2 | `m = 8`, `8 // 8 = 1` | `(8 + 1) % 2 = 1` | `28 + 1 + 2 + 0 = 31` |
| 3 | Thứ tự 400 → 100 → 4 | mỗi tầng một `if` + `return` | `return year % 4 == 0` ở cuối |
| 4 | Phần tử 0 là đệm | `days[month]` | `+1` nếu tháng 2 năm nhuận |
| 5 | Biến `y, m` tăng dần | `m += 1`; nếu `m == 13` thì `m = 1; y += 1` | dừng sau khi xử lý `(y2, m2)` |
| 6 | Số ngày lệch so với thứ Sáu | `(days_since_1971(...) - 1) % 7` | danh sách tên bắt đầu từ `"Friday"` |

## 16. Đáp án tham khảo

- **Bài 1:** 1600 (chia hết 400) và 1996 (chia hết 4, không chia hết 100) nhuận; 1700, 1999 không.
- **Bài 2:** `1 + 2 + 0` → `31`.

In [28]:
def answer_is_leap(year):
    if year % 400 == 0:
        return True
    if year % 100 == 0:
        return False
    return year % 4 == 0

def answer_number_of_days(year, month):
    days = [0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    if month == 2 and answer_is_leap(year):
        return 29
    return days[month]

def answer_count_31_months(y1, m1, y2, m2):
    cnt, y, m = 0, y1, m1
    while (y, m) <= (y2, m2):          # so sánh tuple theo thứ tự từ điển
        if answer_number_of_days(y, m) == 31:
            cnt += 1
        m += 1
        if m == 13:
            m, y = 1, y + 1
    return cnt

def answer_day_of_week(day, month, year):
    names = ["Friday", "Saturday", "Sunday", "Monday", "Tuesday", "Wednesday", "Thursday"]
    offset = days_since_1971(f"{year:04d}-{month:02d}-{day:02d}") - 1
    return names[offset % 7]

assert [answer_is_leap(y) for y in [1600, 1700, 1996, 1999]] == [True, False, True, False]
assert number_of_days_formula(2023, 8) == 31
assert answer_number_of_days(1992, 7) == 31 and answer_number_of_days(2000, 2) == 29 and answer_number_of_days(1900, 2) == 28
assert answer_count_31_months(2023, 1, 2023, 12) == 7
assert answer_count_31_months(2023, 7, 2023, 8) == 2
assert answer_day_of_week(31, 8, 2019) == "Saturday"
assert answer_day_of_week(15, 8, 1993) == "Sunday"
for y in range(1583, 2101):
    assert answer_is_leap(y) == calendar.isleap(y)
    for m in range(1, 13):
        assert answer_number_of_days(y, m) == calendar.monthrange(y, m)[1]
for y in range(1971, 2101, 7):
    for m in range(1, 13):
        assert answer_day_of_week(1, m, y) == calendar.day_name[date(y, m, 1).weekday()]
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


Trong `answer_count_31_months`, `(y, m) <= (y2, m2)` so sánh **tuple theo thứ tự từ điển**: so phần tử đầu, bằng nhau mới so phần tử thứ hai — rất tiện cho ngày tháng.

## 17. Template nhớ nhanh

```python
def is_leap(y):
    return (y % 4 == 0 and y % 100 != 0) or y % 400 == 0

DAYS = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

def number_of_days(year, month):
    if month == 2 and is_leap(year):
        return 29
    return DAYS[month - 1]
```

Câu thần chú: **"4 thì nhuận, 100 thì không, 400 lại nhuận"**.

## 18. Checklist submit

- [ ] Đã xử lý cả 3 tầng: 4, 100, 400?
- [ ] Test riêng 1900 (→ 28) và 2000 (→ 29)?
- [ ] Chỉ số bảng là `month - 1` (hoặc dùng phần tử đệm)?
- [ ] Ngày nhuận **chỉ** cộng cho tháng 2?
- [ ] Ngoặc trong biểu thức `and`/`or` đặt đúng?
- [ ] Tên method đúng: `numberOfDays(self, year, month)`?